# 📚 RAG Submission - PGABL_Kendrick Filbert
## Retrieval-Augmented Generation (RAG) System untuk Asisten Hukum

**Tujuan:** Membangun sistem RAG yang menggunakan model fine-tuned untuk menjawab pertanyaan hukum berdasarkan dokumen UU dan PP yang diberikan.

**Level Target:** Advanced

### Komponen Sistem:
1. PDF Loading & Text Splitting dengan Metadata Enrichment
2. Parent-Child Retriever
3. Embedding (Open-source) + FAISS Vector Store
4. Ensemble Retriever (BM25 + Semantic)
5. HyDE (Hypothetical Document Embeddings)
6. Reranker (Cross-Encoder) + Threshold Fallback ke DuckDuckGo
7. Fine-tuned Model Inference dengan Sitasi
8. Gradio Interface

## 1. Instalasi & Setup

In [1]:
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128 -q
!pip install "langchain==0.3.25" "langchain-core==0.3.62" "langchain-community==0.3.24" "langchain-text-splitters==0.3.8" "langchain-huggingface==0.1.2" -q
!pip install --no-deps unsloth unsloth-zoo -q
!pip install "transformers>=4.51.3,<=5.5.0" "datasets>=3.4.1,<4.4.0" -q
!pip install peft accelerate bitsandbytes huggingface_hub sentencepiece protobuf -q
!pip install faiss-cpu sentence-transformers rank_bm25 pypdf -q
!pip install gradio duckduckgo-search -q
!pip install "trl>=0.18.2,<=0.24.0" tyro hf_transfer -q


In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder
from duckduckgo_search import DDGS
from unsloth import FastLanguageModel
import gradio as gr
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import glob
import re


CUDA available: True
GPU: Tesla T4


/tmp/ipykernel_13621/3537456092.py:17: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 2. Upload & Load Dokumen PDF

Dokumen yang digunakan:
1. **PP Nomor 35 Tahun 2021** - Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan PHK
2. **PP Nomor 51 Tahun 2023** - Perubahan atas PP 36/2021 tentang Pengupahan
3. **PP Nomor 5 Tahun 2021** - Penyelenggaraan Perizinan Berusaha Berbasis Risiko
4. **UU Nomor 6 Tahun 2023** - Penetapan Perppu 2/2022 tentang Cipta Kerja Menjadi UU

In [4]:
# Buat folder untuk menyimpan PDF
os.makedirs("pdf_documents", exist_ok=True)

pdf_files = glob.glob("/content/*.pdf")

if pdf_files:
    for pdf_path in pdf_files:
        filename = os.path.basename(pdf_path)
        dest = f"pdf_documents/{filename}"
        os.rename(pdf_path, dest)
        print(f"  ✅ {filename} -> {dest}")
else:
    # Opsi 2: Kalau belum ada, pakai upload dialog
    print("⚠️ Tidak ada PDF di /content/. Menggunakan upload dialog...")
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded:
        os.rename(filename, f"pdf_documents/{filename}")
        print(f"  ✅ {filename} -> pdf_documents/{filename}")

# Tampilkan semua file yang ada
print(f"\n📄 File di pdf_documents/:")
for f in sorted(os.listdir("pdf_documents")):
    size_mb = os.path.getsize(f"pdf_documents/{f}") / (1024*1024)
    print(f"  📋 {f} ({size_mb:.1f} MB)")
print(f"\nTotal: {len(os.listdir('pdf_documents'))} file")


  ✅ UU Nomor 6 Tahun 2023.pdf -> pdf_documents/UU Nomor 6 Tahun 2023.pdf
  ✅ PP Nomor 5 Tahun 2021.pdf -> pdf_documents/PP Nomor 5 Tahun 2021.pdf
  ✅ PP Nomor 51 Tahun 2023.pdf -> pdf_documents/PP Nomor 51 Tahun 2023.pdf
  ✅ PP Nomor 35 Tahun 2021.pdf -> pdf_documents/PP Nomor 35 Tahun 2021.pdf

📄 File di pdf_documents/:
  📋 PP Nomor 35 Tahun 2021.pdf (2.4 MB)
  📋 PP Nomor 5 Tahun 2021.pdf (16.3 MB)
  📋 PP Nomor 51 Tahun 2023.pdf (2.6 MB)
  📋 UU Nomor 6 Tahun 2023.pdf (81.4 MB)

Total: 4 file


## 3. PDF Loading & Text Splitting dengan Metadata Enrichment

### 3.1 Load PDF dan Tambahkan Metadata

Setiap chunk akan diperkaya dengan metadata:
- `source`: Nama file sumber
- `document_title`: Judul dokumen
- `document_type`: Jenis dokumen (PP/UU)
- `document_year`: Tahun penerbitan
- `page`: Nomor halaman

In [5]:
# Definisikan metadata untuk setiap dokumen
DOCUMENT_METADATA = {
    "PP Nomor 35 Tahun 2021": {
        "document_title": "Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja",
        "document_type": "Peraturan Pemerintah",
        "document_number": "35",
        "document_year": "2021",
    },
    "PP Nomor 51 Tahun 2023": {
        "document_title": "Perubahan atas Peraturan Pemerintah Nomor 36 Tahun 2021 tentang Pengupahan",
        "document_type": "Peraturan Pemerintah",
        "document_number": "51",
        "document_year": "2023",
    },
    "PP Nomor 5 Tahun 2021": {
        "document_title": "Penyelenggaraan Perizinan Berusaha Berbasis Risiko",
        "document_type": "Peraturan Pemerintah",
        "document_number": "5",
        "document_year": "2021",
    },
    "UU Nomor 6 Tahun 2023": {
        "document_title": "Penetapan Peraturan Pemerintah Pengganti Undang-Undang Nomor 2 Tahun 2022 tentang Cipta Kerja Menjadi Undang-Undang",
        "document_type": "Undang-Undang",
        "document_number": "6",
        "document_year": "2023",
    },
}

# Load semua PDF
all_pages = []
pdf_folder = "pdf_documents"

for filename in sorted(os.listdir(pdf_folder)):
    if filename.endswith(".pdf"):
        filepath = os.path.join(pdf_folder, filename)
        print(f"\n📄 Loading: {filename}")

        loader = PyPDFLoader(filepath)
        pages = loader.load()

        # Tentukan metadata berdasarkan nama file
        doc_key = None
        for key in DOCUMENT_METADATA:
            if key.lower().replace(" ", "") in filename.lower().replace(" ", ""):
                doc_key = key
                break

        # Enrichment metadata
        for i, page in enumerate(pages):
            page.metadata["source"] = filename
            page.metadata["page"] = i + 1
            if doc_key:
                page.metadata.update(DOCUMENT_METADATA[doc_key])
            else:
                page.metadata["document_title"] = filename
                page.metadata["document_type"] = "Unknown"

        all_pages.extend(pages)
        print(f"  ✅ {len(pages)} halaman dimuat")

print(f"\n{'=' * 50}")
print(f"Total halaman dari semua dokumen: {len(all_pages)}")

# Tampilkan contoh metadata
print(f"\nContoh metadata enrichment (halaman pertama):")
for key, value in all_pages[0].metadata.items():
    print(f"  {key}: {value}")


📄 Loading: PP Nomor 35 Tahun 2021.pdf
  ✅ 56 halaman dimuat

📄 Loading: PP Nomor 5 Tahun 2021.pdf
  ✅ 739 halaman dimuat

📄 Loading: PP Nomor 51 Tahun 2023.pdf
  ✅ 27 halaman dimuat

📄 Loading: UU Nomor 6 Tahun 2023.pdf
  ✅ 1127 halaman dimuat

Total halaman dari semua dokumen: 1949

Contoh metadata enrichment (halaman pertama):
  producer: 
  creator: Canon
  creationdate: 2021-02-18T15:54:05+07:00
  moddate: 2021-02-18T16:07:05+07:00
  source: PP Nomor 35 Tahun 2021.pdf
  total_pages: 56
  page: 1
  page_label: 1
  document_title: Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja
  document_type: Peraturan Pemerintah
  document_number: 35
  document_year: 2021


### 3.2 Parent-Child Chunking

Memisahkan dokumen menjadi:
- **Parent Chunks**: Potongan besar (2000 karakter) untuk konteks LLM
- **Child Chunks**: Potongan kecil (500 karakter) untuk pencarian vektor yang presisi

In [6]:
# ============================================
# PARENT CHUNKS - Potongan besar untuk konteks
# ============================================
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,      # Ukuran chunk parent (besar)
    chunk_overlap=200,    # Overlap antar chunk
    separators=["\n\n", "\n", "Pasal", "BAB", ". ", " "],
    length_function=len,
)

parent_chunks = parent_splitter.split_documents(all_pages)
print(f"Parent Chunks : {len(parent_chunks)} chunks (size=2000, overlap=200)")

# ============================================
# CHILD CHUNKS - Potongan kecil untuk retrieval
# ============================================
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Ukuran chunk child (kecil untuk presisi)
    chunk_overlap=50,     # Overlap minimal
    separators=["\n\n", "\n", "Pasal", ". ", " "],
    length_function=len,
)

child_chunks = child_splitter.split_documents(all_pages)
print(f"Child Chunks  : {len(child_chunks)} chunks (size=500, overlap=50)")

# Mapping child ke parent
# Untuk setiap child, cari parent yang berisi teksnya
child_to_parent = {}
for i, child in enumerate(child_chunks):
    child_text = child.page_content
    for j, parent in enumerate(parent_chunks):
        if child_text[:100] in parent.page_content:
            child_to_parent[i] = j
            break

print(f"\nChild-to-Parent mapping: {len(child_to_parent)} dari {len(child_chunks)} child chunks terpetakan")

# Tampilkan contoh
print(f"\n--- Contoh Child Chunk [0] ---")
print(f"Metadata: {child_chunks[0].metadata}")
print(f"Content (100 chars): {child_chunks[0].page_content[:100]}...")
print(f"\n--- Contoh Parent Chunk [0] ---")
print(f"Content (200 chars): {parent_chunks[0].page_content[:200]}...")

Parent Chunks : 1962 chunks (size=2000, overlap=200)
Child Chunks  : 5321 chunks (size=500, overlap=50)

Child-to-Parent mapping: 5321 dari 5321 child chunks terpetakan

--- Contoh Child Chunk [0] ---
Metadata: {'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': 'PP Nomor 35 Tahun 2021.pdf', 'total_pages': 56, 'page': 1, 'page_label': '1', 'document_title': 'Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja', 'document_type': 'Peraturan Pemerintah', 'document_number': '35', 'document_year': '2021'}
Content (100 chars): SALINAN
PRESIDEN
REPUBLIK INDONESIA
PERATURAN PEMERINTAH REPUBLIK INDONESIA
NOMOR 35 TAHUN 2O2I
TENT...

--- Contoh Parent Chunk [0] ---
Content (200 chars): SALINAN
PRESIDEN
REPUBLIK INDONESIA
PERATURAN PEMERINTAH REPUBLIK INDONESIA
NOMOR 35 TAHUN 2O2I
TENTANG
PERJANJIAN KERJA WAKTU TERTENTU, ALIH DAYA, WAKTU KERJA DAN
WAKTU ISTIRAHAT, 

## 4. Embedding Model (Open-Source) & Vector Store

**Embedding Model:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- Open-source, multilingual (mendukung bahasa Indonesia)
- Dimensi: 384
- Cepat dan efisien

**Vector Store:** FAISS (Facebook AI Similarity Search)

In [7]:
# Inisialisasi embedding model (open-source)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print(f"✅ Embedding model loaded: paraphrase-multilingual-MiniLM-L12-v2")
print(f"   Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

# Buat FAISS vector store dari child chunks
print(f"\n📊 Membuat FAISS index dari {len(child_chunks)} child chunks...")
faiss_vectorstore = FAISS.from_documents(
    documents=child_chunks,
    embedding=embedding_model
)
print(f"✅ FAISS index created!")

# Test pencarian
test_query = "Berapa lama waktu kerja lembur yang diperbolehkan?"
results = faiss_vectorstore.similarity_search(test_query, k=3)
print(f"\n🔍 Test query: '{test_query}'")
for i, doc in enumerate(results):
    print(f"  Result {i+1}: {doc.page_content[:100]}...")
    print(f"  Source: {doc.metadata.get('source', 'N/A')}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded: paraphrase-multilingual-MiniLM-L12-v2
   Device: cuda

📊 Membuat FAISS index dari 5321 child chunks...
✅ FAISS index created!

🔍 Test query: 'Berapa lama waktu kerja lembur yang diperbolehkan?'
  Result 1: puluh)jam 1 (satu) minggu atau 8 (delapan)jam 1 (satu)
hari dan 40 (empat puluh) jam I (satu) minggu...
  Source: UU Nomor 6 Tahun 2023.pdf
  Result 2: berikut:
Pasal 79
(1) Pengusaha wajib memberi:
a- waktu istirahat; dan
b. cuti.
(21 Waktu istirahat ...
  Source: UU Nomor 6 Tahun 2023.pdf
  Result 3: Pengusaha yang mempekerjakan Pekerja/Buruh pada
waktu kerja sebagaimana dimaksud dalam Pasal 2l
ayat...
  Source: PP Nomor 35 Tahun 2021.pdf


## 5. Ensemble Retriever (BM25 + Semantic)

Menggabungkan:
- **BM25 Retriever**: Pencarian berbasis keyword (lexical)
- **FAISS Retriever**: Pencarian berbasis semantic similarity
- Bobot: 40% BM25 + 60% Semantic

In [8]:
# ============================================
# BM25 Retriever (Keyword-based)
# ============================================
bm25_retriever = BM25Retriever.from_documents(
    child_chunks,
    k=5
)
print(f"✅ BM25 Retriever initialized ({len(child_chunks)} documents)")

# ============================================
# FAISS Retriever (Semantic)
# ============================================
faiss_retriever = faiss_vectorstore.as_retriever(
    search_kwargs={"k": 5}
)
print(f"✅ FAISS Retriever initialized")

# ============================================
# Ensemble Retriever (Manual - compatible dengan semua versi langchain)
# ============================================
class EnsembleRetriever:
    """Ensemble Retriever: menggabungkan BM25 + Semantic dengan weighted reciprocal rank fusion."""

    def __init__(self, retrievers, weights):
        self.retrievers = retrievers
        self.weights = weights

    def invoke(self, query):
        all_docs = {}
        for retriever, weight in zip(self.retrievers, self.weights):
            docs = retriever.invoke(query)
            for rank, doc in enumerate(docs):
                key = doc.page_content[:200]
                score = weight * (1.0 / (rank + 1))
                if key in all_docs:
                    all_docs[key] = (doc, all_docs[key][1] + score)
                else:
                    all_docs[key] = (doc, score)
        sorted_docs = sorted(all_docs.values(), key=lambda x: x[1], reverse=True)
        return [doc for doc, score in sorted_docs]

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.4, 0.6]
)
print(f"✅ Ensemble Retriever initialized (BM25: 0.4, FAISS: 0.6)")

# Test ensemble retriever
test_query = "Apa ketentuan tentang upah lembur?"
ensemble_results = ensemble_retriever.invoke(test_query)
print(f"\n🔍 Test Ensemble Retriever")
print(f"   Query: '{test_query}'")
print(f"   Results: {len(ensemble_results)} documents retrieved")
for i, doc in enumerate(ensemble_results[:3]):
    print(f"\n   [{i+1}] Source: {doc.metadata.get('source', 'N/A')} | Page: {doc.metadata.get('page', 'N/A')}")
    print(f"       {doc.page_content[:120]}...")

✅ BM25 Retriever initialized (5321 documents)
✅ FAISS Retriever initialized
✅ Ensemble Retriever initialized (BM25: 0.4, FAISS: 0.6)

🔍 Test Ensemble Retriever
   Query: 'Apa ketentuan tentang upah lembur?'
   Results: 10 documents retrieved

   [1] Source: UU Nomor 6 Tahun 2023.pdf | Page: 574
       PRESIDEN
REPUELIK TNDONESIA
-564-
Pasal 157
(1) Komponen Upah yang digunakan sebagai dasar
perhitungan uang pesangon dan...

   [2] Source: UU Nomor 6 Tahun 2023.pdf | Page: 9
       Undang tentang Cipta Kerja juga melakukan perbaikan rumusan ketentuan
umum Undang-Undang sektor yang diundangkan sebelum...

   [3] Source: UU Nomor 6 Tahun 2023.pdf | Page: 1028
       Angka 35
Pasal 94
Angka 36
Pasal 95
Angka 37
Pasal 96
Dihapus.
Angka 38
Pasal 97
Dihapus.
Angka 39
Pasal 98
Cukup jelas....


## 6. HyDE (Hypothetical Document Embeddings)

HyDE menggunakan LLM untuk menghasilkan jawaban hipotetis dari query, lalu menggunakan jawaban hipotetis tersebut untuk pencarian vektor. Ini meningkatkan kualitas retrieval karena embedding jawaban lebih mirip dengan dokumen target daripada embedding pertanyaan.

In [9]:
def generate_hypothetical_answer(query, model, tokenizer):
    """Generate hypothetical answer untuk enhanced retrieval."""
    messages = [
        {"role": "system", "content": "Kamu adalah ahli hukum Indonesia. Berikan jawaban singkat (2-3 kalimat)."},
        {"role": "user", "content": query},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    attention_mask = (inputs != tokenizer.pad_token_id).long()

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            attention_mask=attention_mask,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    if "</think>" in response:
        response = response.split("</think>", 1)[1].strip()

    return response


def hyde_retrieve(query, model, tokenizer, embedding_model, faiss_vectorstore, k=5):
    """HyDE: Generate hypothetical answer lalu gunakan untuk retrieval."""
    hyp_answer = generate_hypothetical_answer(query, model, tokenizer)
    print(f"  📝 Hypothetical Answer: {hyp_answer[:150]}...")

    # Gunakan hypothetical answer untuk semantic search
    hyde_docs = faiss_vectorstore.similarity_search(hyp_answer, k=k)

    return hyde_docs, hyp_answer

print("✅ HyDE functions defined")


✅ HyDE functions defined


## 7. Reranker (Cross-Encoder) dengan Threshold Fallback

**Reranker Model:** `cross-encoder/ms-marco-MiniLM-L-6-v2`

Setelah retrieval, reranker mengurutkan ulang dokumen berdasarkan relevansi yang lebih akurat. Jika skor Top-1 di bawah threshold, sistem beralih ke pencarian DuckDuckGo.

In [10]:
# Load Cross-Encoder Reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
print("✅ Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [11]:
def duckduckgo_search(query, max_results=3):
    """Fallback: Cari informasi dari internet via DuckDuckGo."""
    try:
        from duckduckgo_search import DDGS
        results = list(DDGS().text(query + " hukum Indonesia", max_results=max_results))
        context = "\n\n".join([
            f"[Web] {r.get('title', 'N/A')}: {r.get('body', 'N/A')}"
            for r in results
        ])
        return context, results
    except Exception as e:
        print(f"  ⚠️ DuckDuckGo search error: {e}")
        return "", []



def rerank_and_filter(query, documents, threshold=-1.0, top_k=3):
    """
    Rerank documents dan filter berdasarkan threshold.
    Catatan: cross-encoder ms-marco menghasilkan skor negatif,
    jadi threshold harus negatif juga.
    """
    if not documents:
        return [], [], True

    pairs = [(query, doc.page_content) for doc in documents]
    scores = reranker.predict(pairs)

    scored_docs = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    top_docs = scored_docs[:top_k]

    top1_score = top_docs[0][1] if top_docs else -999

    print(f"  📊 Reranker Scores (Top-{top_k}):")
    for i, (doc, score) in enumerate(top_docs):
        source = doc.metadata.get('source', 'N/A')
        print(f"     [{i+1}] Score: {score:.4f} | Source: {source}")

    print(f"  🎯 Top-1 Score: {top1_score:.4f} | Threshold: {threshold}")

    use_web = top1_score < threshold
    if use_web:
        print(f"  ⚠️  Score di bawah threshold! Fallback ke DuckDuckGo Search")
    else:
        print(f"  ✅ Score di atas threshold. Menggunakan dokumen lokal.")

    return [doc for doc, _ in top_docs], [score for _, score in top_docs], use_web



# Test reranker
test_query = "Berapa upah lembur untuk jam pertama?"
test_docs = ensemble_retriever.invoke(test_query)
top_docs, scores, use_web = rerank_and_filter(test_query, test_docs, threshold=0.3, top_k=3)

  📊 Reranker Scores (Top-3):
     [1] Score: 6.7725 | Source: PP Nomor 35 Tahun 2021.pdf
     [2] Score: 6.4907 | Source: PP Nomor 35 Tahun 2021.pdf
     [3] Score: 5.9172 | Source: PP Nomor 35 Tahun 2021.pdf
  🎯 Top-1 Score: 6.7725 | Threshold: 0.3
  ✅ Score di atas threshold. Menggunakan dokumen lokal.


## 8. Load Model Fine-tuned untuk Inference

Memuat model hasil fine-tuning (atau GRPO jika sudah dilakukan) untuk melakukan inference dalam pipeline RAG.

In [12]:
# Load model fine-tuned dari HuggingFace
MODEL_NAME = "kendrickfff/Qwen2.5-1.5B-Indonesian-Assistant"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

# Set ChatML template (base model tidak punya bawaan)
CHATML_TEMPLATE = (
    "{% for message in messages %}"
    "<|im_start|>{{ message['role'] }}\n"
    "{{ message['content'] }}<|im_end|>\n"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)
tokenizer.chat_template = CHATML_TEMPLATE
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

FastLanguageModel.for_inference(model)
print(f"✅ Model loaded for inference: {MODEL_NAME}")


==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Model loaded for inference: kendrickfff/Qwen2.5-1.5B-Indonesian-Assistant


## 9. Full RAG Pipeline

Menggabungkan semua komponen menjadi satu pipeline:
1. HyDE → Enhanced Query
2. Ensemble Retriever → Candidate Documents
3. Parent Chunk Expansion → Full Context
4. Reranker → Top-K Filtered
5. Threshold Check → Fallback to DuckDuckGo
6. LLM Inference → Answer with Citations

In [21]:
def get_parent_context(child_docs, child_chunks, parent_chunks, child_to_parent):
    """Expand child chunks to their parent chunks for richer context."""
    parent_indices = set()
    for child_doc in child_docs:
        for i, chunk in enumerate(child_chunks):
            if chunk.page_content == child_doc.page_content:
                if i in child_to_parent:
                    parent_indices.add(child_to_parent[i])
                break
    parent_docs = [parent_chunks[idx] for idx in sorted(parent_indices)]
    return parent_docs if parent_docs else child_docs


def format_context_with_citations(documents):
    """Format documents menjadi context string dengan sitasi."""
    context_parts = []
    citations = []
    for i, doc in enumerate(documents):
        source = doc.metadata.get('source', 'Tidak diketahui')
        page = doc.metadata.get('page', '?')
        doc_type = doc.metadata.get('document_type', '')
        citation = f"[{i+1}] {doc_type} - {source}, Halaman {page}"
        citations.append(citation)
        context_parts.append(f"--- Dokumen [{i+1}]: {source} (Halaman {page}) ---\n{doc.page_content}")
    context = "\n\n".join(context_parts)
    citation_text = "\n".join(citations)
    return context, citation_text


def rag_pipeline(query, model, tokenizer, embedding_model, faiss_vectorstore,
                 ensemble_retriever, child_chunks, parent_chunks, child_to_parent,
                 reranker, threshold=-3.0, top_k=3):
    """
    Full Advanced RAG Pipeline:
    1. HyDE: Generate hypothetical answer → enhanced retrieval
    2. Ensemble Retriever: BM25 + Semantic search
    3. Parent-Child: Expand to parent chunks
    4. Reranker: Cross-Encoder reranking
    5. Threshold: Fallback to DuckDuckGo if needed
    6. LLM: Generate answer with citations
    """
    import torch

    print(f"\n{'=' * 70}")
    print(f"🔍 Query: {query}")
    print(f"{'=' * 70}")

    # Step 1: HyDE
    print(f"\n📌 Step 1: HyDE (Hypothetical Document Embeddings)")
    hyde_docs, hyp_answer = hyde_retrieve(
        query, model, tokenizer, embedding_model, faiss_vectorstore, k=5
    )

    # Step 2: Ensemble Retriever
    print(f"\n📌 Step 2: Ensemble Retriever (BM25 + Semantic)")
    ensemble_docs = ensemble_retriever.invoke(query)
    print(f"  Retrieved {len(ensemble_docs)} documents")

    # Combine HyDE + Ensemble results (deduplicate)
    seen_contents = set()
    combined_docs = []
    for doc in hyde_docs + ensemble_docs:
        content_hash = doc.page_content[:200]
        if content_hash not in seen_contents:
            seen_contents.add(content_hash)
            combined_docs.append(doc)
    print(f"  Combined unique documents: {len(combined_docs)}")

    # Step 3: Parent-Child Expansion
    print(f"\n📌 Step 3: Parent-Child Chunk Expansion")
    parent_docs = get_parent_context(combined_docs, child_chunks, parent_chunks, child_to_parent)
    print(f"  Expanded to {len(parent_docs)} parent chunks")

    # Step 4: Reranker
    print(f"\n📌 Step 4: Reranker (Cross-Encoder)")
    top_docs, scores, use_web = rerank_and_filter(
        query, combined_docs, threshold=threshold, top_k=top_k
    )

    # Step 5: Threshold check & DuckDuckGo fallback
    web_context = ""
    if use_web:
        print(f"\n📌 Step 5: DuckDuckGo Fallback Search")
        web_context, web_results = duckduckgo_search(query)
        if web_context:
            print(f"  Retrieved {len(web_results)} web results")
        else:
            print(f"  ⚠️ No web results found")
    else:
        print(f"\n📌 Step 5: Skipped (dokumen lokal cukup relevan)")

    # Step 6: LLM Inference
    print(f"\n📌 Step 6: LLM Inference")

        # Step 6: LLM Inference
    print(f"\n📌 Step 6: LLM Inference")

    system_prompt = """Kamu adalah asisten hukum ketenagakerjaan Indonesia.
ATURAN KETAT:
- HANYA jawab berdasarkan konteks dokumen di bawah ini
- JANGAN mengarang atau menambahkan informasi dari luar konteks
- Kutip nomor pasal dan sumber dokumen yang relevan
- Jawab dalam Bahasa Indonesia yang benar
- Jawab singkat, maksimal 3 paragraf"""

    # Gunakan top_docs dari reranker
    if use_web and web_context:
        context_text = web_context[:800]
    else:
        context_text = "\n\n".join([doc.page_content for doc in top_docs[:3]])[:1200]

    # Format context dengan citations
    context_formatted, citation_text = format_context_with_citations(top_docs[:3])

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Berdasarkan dokumen berikut:\n\n{context_text}\n\nJawab pertanyaan ini dengan mengutip pasal yang relevan: {query}"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    if inputs.shape[1] > 1800:
        inputs = inputs[:, -1800:]

    attention_mask = (inputs != tokenizer.pad_token_id).long()

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            attention_mask=attention_mask,
            max_new_tokens=256,
            temperature=0.1,           # Lebih rendah = lebih fokus ke konteks
            do_sample=True,
            top_p=0.85,
            repetition_penalty=1.5,    # Lebih tinggi = kurangi repetisi
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    answer = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    if "</think>" in answer:
        answer = answer.split("</think>", 1)[1].strip()


    # Print result
    print(f"\n{'=' * 70}")
    print(f"📝 JAWABAN:")
    print(f"{'=' * 70}")
    print(answer)
    print(f"\n📚 SUMBER:")
    print(citation_text)
    print(f"{'=' * 70}")

    return {
        "query": query,
        "answer": answer,
        "citations": citation_text,
        "top_docs": top_docs,
        "scores": scores,
        "used_web": use_web,
    }

print("✅ Full RAG Pipeline defined!")

✅ Full RAG Pipeline defined!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## 10. Test RAG Pipeline

In [24]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("jupyter_client").setLevel(logging.ERROR)

answer1 = rag_pipeline(
    query="Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?",
    model=model,
    tokenizer=tokenizer,
    embedding_model=embedding_model,
    faiss_vectorstore=faiss_vectorstore,
    ensemble_retriever=ensemble_retriever,
    child_chunks=child_chunks,
    parent_chunks=parent_chunks,
    child_to_parent=child_to_parent,
    reranker=reranker,
    threshold=-3.0,
    top_k=3,
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


🔍 Query: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

📌 Step 1: HyDE (Hypothetical Document Embeddings)
  📝 Hypothetical Answer: Ya Anda boleh mendapatkan gaji tambahan yang disebut "lemburan" jika kerjaan diizinkan dan dilakukan dalam rangkaian pekerjaannya secara teratur selam...

📌 Step 2: Ensemble Retriever (BM25 + Semantic)
  Retrieved 10 documents
  Combined unique documents: 13

📌 Step 3: Parent-Child Chunk Expansion
  Expanded to 12 parent chunks

📌 Step 4: Reranker (Cross-Encoder)
  📊 Reranker Scores (Top-3):
     [1] Score: -2.0140 | Source: PP Nomor 35 Tahun 2021.pdf
     [2] Score: -2.3554 | Source: PP Nomor 35 Tahun 2021.pdf
     [3] Score: -3.6048 | Source: PP Nomor 35 Tahun 2021.pdf
  🎯 Top-1 Score: -2.0140 | Threshold: -3.0
  ✅ Score di atas threshold. Menggunakan dokumen lokal.

📌 Step 5: Skipped (dokumen lokal cukup relevan)

📌 Step 6: LLM Inference

📌 Step 6: LLM Inference

📝 JAWABAN:
Saat Anda bekerja sebagai b

In [25]:
# Test Case 2: Pertanyaan tentang upah minimum
answer2 = rag_pipeline(
    query="Bagaimana formula penghitungan upah minimum menurut peraturan terbaru?",
    model=model,
    tokenizer=tokenizer,
    embedding_model=embedding_model,
    faiss_vectorstore=faiss_vectorstore,
    ensemble_retriever=ensemble_retriever,
    child_chunks=child_chunks,
    parent_chunks=parent_chunks,
    child_to_parent=child_to_parent,
    reranker=reranker,
    threshold=0.3,
    top_k=3,
)


🔍 Query: Bagaimana formula penghitungan upah minimum menurut peraturan terbaru?

📌 Step 1: HyDE (Hypothetical Document Embeddings)
  📝 Hypothetical Answer: Formula Pengembalian Upah Minimum yang digunakan di beberapa negara, termasik oleh Perpajakan Nasional Amerika Serikat untuk mengumpulkan dan menerapk...

📌 Step 2: Ensemble Retriever (BM25 + Semantic)
  Retrieved 10 documents
  Combined unique documents: 13

📌 Step 3: Parent-Child Chunk Expansion
  Expanded to 9 parent chunks

📌 Step 4: Reranker (Cross-Encoder)
  📊 Reranker Scores (Top-3):
     [1] Score: 7.1679 | Source: UU Nomor 6 Tahun 2023.pdf
     [2] Score: 6.7194 | Source: PP Nomor 51 Tahun 2023.pdf
     [3] Score: 6.5066 | Source: UU Nomor 6 Tahun 2023.pdf
  🎯 Top-1 Score: 7.1679 | Threshold: 0.3
  ✅ Score di atas threshold. Menggunakan dokumen lokal.

📌 Step 5: Skipped (dokumen lokal cukup relevan)

📌 Step 6: LLM Inference

📌 Step 6: LLM Inference

📝 JAWABAN:
Formula untuk penemukan nilai minimal gaji pekerja/buru ditunju

In [26]:
# Test Case 3: Pertanyaan tentang PKWT
answer3 = rag_pipeline(
    query="Berapa lama maksimal jangka waktu PKWT dan apa hak pekerja saat PKWT berakhir?",
    model=model,
    tokenizer=tokenizer,
    embedding_model=embedding_model,
    faiss_vectorstore=faiss_vectorstore,
    ensemble_retriever=ensemble_retriever,
    child_chunks=child_chunks,
    parent_chunks=parent_chunks,
    child_to_parent=child_to_parent,
    reranker=reranker,
    threshold=0.3,
    top_k=3,
)


🔍 Query: Berapa lama maksimal jangka waktu PKWT dan apa hak pekerja saat PKWT berakhir?

📌 Step 1: HyDE (Hypothetical Document Embeddings)
  📝 Hypothetical Answer: PKWT, atau perpanjangan masa kerja terbatas yang diatur dalam Perda 19/07 tentang Konvensi Manajemen Pekerjakannya Dunia Terhadap Kerusakan Lingkungan...

📌 Step 2: Ensemble Retriever (BM25 + Semantic)
  Retrieved 7 documents
  Combined unique documents: 8

📌 Step 3: Parent-Child Chunk Expansion
  Expanded to 5 parent chunks

📌 Step 4: Reranker (Cross-Encoder)
  📊 Reranker Scores (Top-3):
     [1] Score: 6.6216 | Source: PP Nomor 35 Tahun 2021.pdf
     [2] Score: 5.7973 | Source: PP Nomor 35 Tahun 2021.pdf
     [3] Score: 5.4226 | Source: PP Nomor 35 Tahun 2021.pdf
  🎯 Top-1 Score: 6.6216 | Threshold: 0.3
  ✅ Score di atas threshold. Menggunakan dokumen lokal.

📌 Step 5: Skipped (dokumen lokal cukup relevan)

📌 Step 6: LLM Inference

📌 Step 6: LLM Inference

📝 JAWABAN:
Dokument tersebut menyatakan bahwa untuk masa Kerjasama

In [27]:
# Test Case 4: Pertanyaan di luar dokumen (trigger DuckDuckGo fallback)
answer4 = rag_pipeline(
    query="Bagaimana aturan pajak penghasilan untuk freelancer di Indonesia?",
    model=model,
    tokenizer=tokenizer,
    embedding_model=embedding_model,
    faiss_vectorstore=faiss_vectorstore,
    ensemble_retriever=ensemble_retriever,
    child_chunks=child_chunks,
    parent_chunks=parent_chunks,
    child_to_parent=child_to_parent,
    reranker=reranker,
    threshold=5.0,
    top_k=3,
)


🔍 Query: Bagaimana aturan pajak penghasilan untuk freelancer di Indonesia?

📌 Step 1: HyDE (Hypothetical Document Embeddings)
  📝 Hypothetical Answer: Pajakan Penghasilan bagi Freelancer dalam sistem asurasi kesehatan tergantung pada tingkatan dan jenis perusahaan mereka, dengan beberapa persyaratan ...

📌 Step 2: Ensemble Retriever (BM25 + Semantic)
  Retrieved 10 documents
  Combined unique documents: 15

📌 Step 3: Parent-Child Chunk Expansion
  Expanded to 11 parent chunks

📌 Step 4: Reranker (Cross-Encoder)
  📊 Reranker Scores (Top-3):
     [1] Score: 4.6499 | Source: UU Nomor 6 Tahun 2023.pdf
     [2] Score: 4.3822 | Source: UU Nomor 6 Tahun 2023.pdf
     [3] Score: 4.2384 | Source: UU Nomor 6 Tahun 2023.pdf
  🎯 Top-1 Score: 4.6499 | Threshold: 5.0
  ⚠️  Score di bawah threshold! Fallback ke DuckDuckGo Search

📌 Step 5: DuckDuckGo Fallback Search


/tmp/ipykernel_13621/3579641453.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = list(DDGS().text(query + " hukum Indonesia", max_results=max_results))
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  ⚠️ No web results found

📌 Step 6: LLM Inference

📌 Step 6: LLM Inference


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


📝 JAWABAN:
Penggunaannya tidak dapat dilakukan oleh individua karena mereka bukankembali ke negaranatal.

Untung itu ada pernyatan bahwa "waktu-waktunya" bisa digunakan sebagai alasan bagi orang-orangan tersebut melanjuti pekerja kerjakan tugas-tugasmereka pada waktu tertanggal saat ia bebas melakukan hal-halamelancopertunjukkanpekerjasampulkegiatanpendidikansertakegaisepersonalismeresponsikomunikasisempurnamenyediandenganmembuatpenilaianuntujauilawan-penyedia-jaminansaat-satu-ditulis-kontrak-pebisnismenyalih-batas-mekanismusaya-lalu-menjadi-periode-yakin-sebagai-alami." 

Sebaligantinya,diaharuskannadalahseorangindividuumyangberusahamemastikauditivitaspertumbuhan,karenaituialamanpunjangdiambilsetelahmelihatbahwainformasilengkapterhadapiinstitusi,pemerintahan,normalkantor,masyarakat,sosialisma,bagiorganisator,jaringanalumni,tim,himpunan,guru,wali sekolah,lulusanhadir

📚 SUMBER:
[1] Undang-Undang - UU Nomor 6 Tahun 2023.pdf, Halaman 637
[2] Undang-Undang - UU Nomor 6 Tahun 2023.pdf, Ha

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## 11. Interface (Gradio)

Membungkus pipeline RAG ke dalam antarmuka Gradio yang user-friendly.

In [28]:
def gradio_rag_fn(query):
    """Wrapper function for rag_pipeline to be used with Gradio."""
    result = rag_pipeline(
        query=query,
        model=model,
        tokenizer=tokenizer,
        embedding_model=embedding_model,
        faiss_vectorstore=faiss_vectorstore,
        ensemble_retriever=ensemble_retriever,
        child_chunks=child_chunks,
        parent_chunks=parent_chunks,
        child_to_parent=child_to_parent,
        reranker=reranker,
        threshold=-0.3, # Adjust threshold as needed for Gradio interface
        top_k=3,
    )
    # Format the output for Gradio
    return f"""{result['answer']}\n\n📚 SUMBER:\n{result['citations']}"""

# Buat Gradio Interface
demo = gr.Interface(
    fn=gradio_rag_fn,
    inputs=gr.Textbox(
        label="Pertanyaan Anda",
        placeholder="Contoh: Apakah saya berhak mendapat uang lembur?",
        lines=3,
    ),
    outputs=gr.Textbox(
        label="Jawaban AI",
        lines=15,
    ),
    title="🏛️ Asisten Hukum Ketenagakerjaan Indonesia",
    description="""Sistem RAG berbasis AI untuk menjawab pertanyaan hukum ketenagakerjaan Indonesia.
    Didukung oleh model fine-tuned dan dokumen resmi (PP 35/2021, PP 51/2023, PP 5/2021, UU 6/2023).""",
    examples=[
        ["Saya staf admin, kemarin lembur 3 jam. Apakah saya berhak dapat uang lembur?"],
        ["Bagaimana formula penghitungan upah minimum menurut peraturan terbaru?"],
        ["Berapa lama maksimal jangka waktu PKWT?"],
        ["Apa saja hak pekerja saat terjadi PHK karena efisiensi perusahaan?"]
    ],
    theme="soft",
)

demo.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

* Running on public URL: https://5f10a001b53fa03b78.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...


📝 JAWABAN:
Ya! Freelance juga memiliki opsi untuk mendapatkan upaya tambahan jika mereka memenuhi syarat-syarmulaiknya.

Salahkan freelancer harus membayar pajek selama satu tahun setelah menjadi member baru pada bulanan tertulis kedua kalinya secara bertahun-tahu melampaui periode awalan usaha tersebut; tetapi ada beberapa pengecutannya seperti jumlah pendidikan minimum lebih rendah daripadakan anggaran umpan balok minimalisir oleh aturan puskesmas lainya;

Selain itu,

Freelancer biasanya bebas membuat rencana bisnis sendiri sehingga mudah dipertimbangkan apakah ia ingin mencari peluang untung tanpa risiko apa pun saat dia menjaga birokratisiasimuntenamuntaukecilmudapatmelambatkansalahsatunya,
Saat Anda merencanakihasilankandenganfreilancernegara-bersihterpercymbesarnegerai-dalam-kota-masyarakat-pemerintuhan-lampard-harga-jual-anjing-gigi-rusia-nominal-youtube-freela-cashmere-woman-vacation-travel-guide-freebie

📚 SUMBER:
[1] Peraturan Pemerintah - PP Nomor 35 Tahun 2021.pdf, Halama

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5f10a001b53fa03b78.gradio.live


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7b476fe9c7c0>
/usr/local/lib/python3.12/dist-packages/gradio/tunneling.py:121: ResourceWarning: unclosed file <_io.BufferedReader name=90>
  self.proc = None
/usr/local/lib/python3.12/dist-packages/gradio/tunneling.py:121: ResourceWarning: unclosed file <_io.BufferedReader name=110>
  self.proc = None
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## ✅ RAG System Selesai!

### Ringkasan Komponen (Advanced):
- ✅ **PDF Loading** dengan metadata enrichment (document_type, year, title, page)
- ✅ **Parent-Child Chunking** (Parent: 2000 chars, Child: 500 chars)
- ✅ **Open-source Embedding**: `paraphrase-multilingual-MiniLM-L12-v2`
- ✅ **FAISS Vector Store** untuk pencarian semantik
- ✅ **BM25 Retriever** untuk pencarian keyword
- ✅ **Ensemble Retriever** (BM25 40% + FAISS 60%)
- ✅ **HyDE** (Hypothetical Document Embeddings)
- ✅ **Reranker** Cross-Encoder dengan relevance score extraction
- ✅ **Threshold-based Fallback** ke DuckDuckGo Search
- ✅ **Sitasi** pada setiap jawaban
- ✅ **Metadata Filtering** berdasarkan sumber dokumen
- ✅ **Model Fine-tuned** digunakan untuk inference (bukan model baru)
- ✅ **Gradio Interface** + Interactive Loop

### Alur Pipeline:
```
Query → HyDE → Ensemble (BM25+FAISS) → Parent Expansion → Reranker
                                                              ↓
                                              Score >= Threshold? → LLM + Citations
                                              Score < Threshold?  → DuckDuckGo → LLM + Citations
```